# sol04: Stack Samples to Trace Events

Contains:
- the same scenario as `04_mock`
- one complete reference implementation
- grading tests


In [ ]:
from collections import defaultdict
from typing import Any

TRACE_SAMPLES = [
    {"ts": 0, "stack": ["main"]},
    {"ts": 1, "stack": ["main", "load"]},
    {"ts": 3, "stack": ["main", "load", "parse"]},
    {"ts": 5, "stack": ["main", "render"]},
]

NOOP_SAMPLES = [
    {"ts": 10, "stack": ["main"]},
    {"ts": 11, "stack": ["main"]},
]


In [ ]:
def samples_to_events(samples: list[dict[str, Any]], close_final: bool = False) -> list[dict[str, Any]]:
    if not isinstance(samples, list):
        raise ValueError("samples_must_be_list")
    if not samples:
        return []

    events: list[dict[str, Any]] = []
    prev_ts: int | None = None
    old_stack: list[str] = []

    for sample in samples:
        if "ts" not in sample or "stack" not in sample:
            raise ValueError("sample_missing_required_fields")
        ts = sample["ts"]
        new_stack = sample["stack"]
        if not isinstance(ts, int):
            raise ValueError("timestamp_must_be_int")
        if not isinstance(new_stack, list) or any(not isinstance(frame, str) for frame in new_stack):
            raise ValueError("stack_must_be_list_of_strings")
        if prev_ts is not None and ts < prev_ts:
            raise ValueError("timestamps_must_be_non_decreasing")
        prev_ts = ts

        prefix = 0
        while prefix < len(old_stack) and prefix < len(new_stack) and old_stack[prefix] == new_stack[prefix]:
            prefix += 1

        for fn in reversed(old_stack[prefix:]):
            events.append({"type": "end", "fn": fn, "ts": ts})
        for fn in new_stack[prefix:]:
            events.append({"type": "start", "fn": fn, "ts": ts})

        old_stack = list(new_stack)

    if close_final and samples:
        flush_ts = samples[-1]["ts"] + 1
        for fn in reversed(old_stack):
            events.append({"type": "end", "fn": fn, "ts": flush_ts})

    return events


def longest_running_function(samples: list[dict[str, Any]]) -> tuple[str, int]:
    events = samples_to_events(samples, close_final=True)
    if not events:
        raise ValueError("no_events")

    open_frames: dict[str, int] = {}
    durations: defaultdict[str, int] = defaultdict(int)
    for event in events:
        fn = event["fn"]
        ts = event["ts"]
        if event["type"] == "start":
            open_frames[fn] = ts
            continue
        start_ts = open_frames.pop(fn, None)
        if start_ts is None:
            continue
        durations[fn] += ts - start_ts

    if not durations:
        raise ValueError("no_durations")

    winner = sorted(durations.items(), key=lambda kv: (-kv[1], kv[0]))[0]
    return winner


In [ ]:
def run_exam04_tests() -> None:
    expected = [
        {"type": "start", "fn": "main", "ts": 0},
        {"type": "start", "fn": "load", "ts": 1},
        {"type": "start", "fn": "parse", "ts": 3},
        {"type": "end", "fn": "parse", "ts": 5},
        {"type": "end", "fn": "load", "ts": 5},
        {"type": "start", "fn": "render", "ts": 5},
        {"type": "end", "fn": "render", "ts": 6},
        {"type": "end", "fn": "main", "ts": 6},
    ]
    events = samples_to_events(TRACE_SAMPLES, close_final=True)
    assert events == expected

    # unchanged stack should not emit transitions
    events = samples_to_events(NOOP_SAMPLES, close_final=True)
    assert events == [
        {"type": "start", "fn": "main", "ts": 10},
        {"type": "end", "fn": "main", "ts": 12},
    ]

    fn, duration = longest_running_function(TRACE_SAMPLES)
    assert fn == "main"
    assert duration == 6

    # invalid timestamps
    try:
        samples_to_events([{"ts": 2, "stack": ["main"]}, {"ts": 1, "stack": ["main"]}], close_final=True)
        raise AssertionError("Expected timestamp validation failure")
    except ValueError as exc:
        assert "timestamps_must_be_non_decreasing" in str(exc)

    print("04_mock tests passed")


run_exam04_tests()
